### Question 1: Demand Surge Identification

> Which product categories experienced the greatest sales surge in the 7 days before hurricane landfall compared to the prior baseline period? Quantify the percentage increase and visualize the daily trend for the top 5 categories.

**Requires:** Cleaning transaction data (deduplication, fixing SKUs, fixing timestamps), grouping by category, computing baseline vs. surge periods, line chart or bar chart.

---


# Cleaning the data first

In [ ]:
import seaborn as sns
import pandas as pd

# set up cleaning log
DECISIONS = []
def log(step, decision, rows_affected):
    DECISIONS.append({'step': step, 'decision': decision, 'rows': rows_affected})
    print(f'[{step}] {decision} ({rows_affected} row(s))')

def show_log():
    return pd.DataFrame(DECISIONS)



In [ ]:
# cleaning methods

def better_consolidate(df: pd.DataFrame, mapping: dict, column: str, target_column: str) -> pd.DataFrame:
    out = df.copy()
    # .map() replaces values found in the dictionary, leaving unmapped values as NaN,
    # then .fillna() puts the original column value back if no mapping exists.
    out[target_column] = out[column].map(mapping).fillna(out[column])
    return out

def remove_duplicates(df):
    df = df.copy()
    after = df.drop_duplicates()
    removed = len(df) - len(after)
    log('duplicates', 'dropped exact duplicate rows', removed)
    return after


def clean_unit_price(df, column ='unit_price'):
    before = df.copy()
    df = df.copy()
    df[column] = df[column].astype(str).str.replace('$', '', regex=False).str.strip()
    df[column] = pd.to_numeric(df[column], errors='coerce')
    log('clean unit prices', 'strip and to numeric', len(df.compare(before)))
    return df

def neg_quant(df, column = 'quantity'):
    before = df.copy()
    df = df.copy()
    if column in df.columns:
        df = df[df[column]>=0]
    removed = len(before) - len(df)
    log('negative quantity', 'drop negative quantities', removed)
    return df

def clean_store_id(df, column = 'store_id'):
    before = df.copy()
    df = df.copy()
    if column in df.columns:
        df[column] = df[column] = df[column].astype(str).str.strip().str.upper()
    log('clean store id', 'strip and upper', len(df.compare(before)))
    return df


def parse_timestamp(df, column = 'timestamp'):
    df = df.copy()
    rows_parsed = 0
    unparsed_count = 0
    if column in df.columns:
        values = df[column].astype('string').str.strip()
        parsed = pd.to_datetime(values, format='mixed', errors='coerce')
        unparsed_count = int((values.notna() & values.ne('') & parsed.isna()).sum())
        df[column] = parsed
        rows_parsed = int(parsed.notna().sum())
    log(
        'parse timestamps',
        f'parsed mixed formats; {unparsed_count} non-empty value(s) were unparseable',
        rows_parsed,
    )
    return df

def lowercase_category_names(df, column='category'):
    df = df.copy()
    rows_changed = 0
    if column in df.columns:
        before = df[column]
        df[column] = before.str.lower()
        rows_changed = int((before.ne(df[column]) & before.notna()).sum())
    log('lowercase category names', 'convert category names to lowercase', rows_changed)
    return df

def standardize_category_names(df, column='category'):
    mappings = {
        'emergency supplies': 'emergency',
        'breakfast foods': 'breakfast',
        'groceries': 'grocery',
        'household essentials': 'household',
        'baby products': 'baby',
        'alcoholic beverages': 'alcohol',
        'health & wellness': 'health',
        'games & entertainment': 'entertainment'
    }
    df = df.copy()
    rows_changed = 0
    if column in df.columns:
        before = df[column]
        df = better_consolidate(df, mappings, "category", "category")
        rows_changed = int((before.ne(df[column]) & before.notna()).sum())
    log('consolidate category names', 'consolidate some categories that meant the same thing', rows_changed)
    return df

def add_date(df):
    df['date'] = df['timestamp'].dt.normalize()

In [ ]:
# run cleaning steps

path = '../data/walmart_transactions.csv'
df = pd.read_csv(path)

df = remove_duplicates(df)
df = clean_unit_price(df)
df = neg_quant(df)
df = clean_store_id(df)
df = lowercase_category_names(df)
df = parse_timestamp(df)
df = standardize_category_names(df)
add_date(df)

show_log()
print(df.shape)

cleaned_transactions = df

df.head()

# Group by category and calculate baseline and surge

In [ ]:
category_count = df.groupby(["date", "category"]).agg(
    count=('quantity', 'sum')
).reset_index()

unique_dates = sorted(category_count['date'].unique())
first_15_dates = unique_dates[:15]

# Calculate baseline average count per category over the first 15 days
baseline_df = (
    category_count[category_count['date'].isin(first_15_dates)]
    .groupby('category')['count']
    .mean()
    .reset_index(name='baseline_avg')
)

# 7 days before hurricane
target_start = pd.to_datetime('2024-09-19')
target_end = pd.to_datetime('2024-09-25')
target_dates = pd.date_range(start=target_start, end=target_end)

# Calculate average count per category during this 7-day window
period_df = (
    category_count[category_count['date'].isin(target_dates)]
    .groupby('category')['count']
    .mean()
    .reset_index(name='week_before_landfall_avg')
)

comparison_df = pd.merge(baseline_df, period_df, on='category')
comparison_df['pct_increase'] = (
    (comparison_df['week_before_landfall_avg'] - comparison_df['baseline_avg'])
    / comparison_df['baseline_avg']
) * 100

comparison_df = comparison_df.sort_values(by="pct_increase", ascending=False)

comparison_df


The breakfast category had the greatest percent increase, with over double of the emergency category...

# Create plot of top five categories

In [ ]:
from matplotlib import pyplot as plt


top_5_categories = ["emergency", "breakfast", "electronics", "entertainment", "alcohol"]
df_filtered = df[df['category'].isin(top_5_categories)]
category_count = df_filtered.groupby(["date", "category"]).agg(
    count=('category', 'count')
).reset_index()

fig, ax = plt.subplots(figsize=(12, 6))
sns.lineplot(
    data=category_count,
    x='date',
    y='count',
    hue='category',
    marker='o'
)

plt.title('Transactions per category over time', fontsize=14, fontweight='bold')
plt.xlabel('Date', fontsize=12)
plt.ylabel('Count', fontsize=12)
plt.xticks(rotation=45)
plt.legend(title='Category', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()

# 4. Display the chart
plt.show()


The breakfast category's transactions experienced the greatest sales surge leading up to the hurricane's landfall.

### Question 2: The Pop-Tarts Effect (SKU Consolidation)

> After standardizing all Pop-Tart SKU variants into a single product, what is the true daily sales volume? How does the uncleaned (fragmented) view compare to the cleaned (consolidated) view? What business decisions would differ between the two views?

**Requires:** Building an SKU mapping table, consolidating fragmented products, side-by-side visualization (before vs. after cleaning), written analysis of business impact.

---

# Create SKU mapping

In [ ]:
import pandas as pd
def create_mappings(from_list: list[str], to: str) -> dict[str, str]:
    d = {}
    for fr in from_list:
        d[fr] = to
    return d

CANON = {}
# |= merges dictionaries
CANON |= create_mappings(["SKU#459812", "POPTART-STRAW", "PT_STRAWBERRY"],
                         "PT-12")
CANON |= create_mappings(["WTR-24", "WATER24PK", "WTR_24PK"],
                         "WTR-24")
CANON |= create_mappings(['BG-MONO', 'MONOPOLY-01', 'GAME-MONO'],
                         'BG-MONO')
CANON |= create_mappings(['BEER-24', 'LTBEER24'],
                         'BEER-24')
CANON |= create_mappings(['BATT-100', 'BAT-AA4', 'BATT_AA_4PK' ],
                         'BATT-AA')
CANON |= create_mappings(['FLASH-01','LED-FLASH'],
                         'LED-FLASH')
CANON |= create_mappings(['CARDS-01','PLAYCARD-STD'],
                         'PLAYCARD-STD')
CANON |= create_mappings(['STREAM-01','ROKU-STD'],
                         'STREAM-01')
CANON |= create_mappings(['SOUP-CAN','CNSOUP-01',],
                         'CNSOUP-01')
CANON |= create_mappings(['BREAD-WH', 'WHITEBREAD'],
                         'BREAD-24')
CANON |= create_mappings(['PB-16' , 'PNUTBTR-16'],
                         'PB-16')
CANON |= create_mappings(['CNDL-EM' ,'CANDLE-EMRG'],
                         'CANDLE-EM' )
CANON |= create_mappings(['FAID-01', 'FIRSTAID-KIT'],
                         'FAID-01' )
CANON |= create_mappings(['CHRG-PORT' , 'PORTCHARGER'],
                         'CHRG-PORT' )
CANON |= create_mappings(['CHIPS-VP', 'CHIPVARIETY'],
                         'CHIPS-VP' )
CANON |= create_mappings(['PTOWEL-6', 'PAPERTOWEL6'],
                         'PTOWEL-6')
CANON |= create_mappings(['TP-12', 'TOILETPAPER12'],
                         'TP-12')
CANON |= create_mappings(['DIAP-S3', 'DIAPERS-3'],
                         'DIAP-S3')
CANON |= create_mappings(['BFORM-01', 'BABYFORMULA'],
                         'BFORM-01')
CANON |= create_mappings(['TARP-HD', 'HDTARP-01'],
                         'TARP-HD')


# Consolidate on the product catalog

In [ ]:
df = pd.read_csv('../data/product_catalog.csv')

def consolidate(frame):
    out = frame.copy()
    out['canonical_sku'] = out.apply(apply_canonical_sku, axis=1)
    return out

def apply_canonical_sku(row):
    if row['sku'] in CANON:
        return CANON[row['sku']]
    return row['sku']

consolidated = consolidate(df)
consolidated

# Consolidate on the transaction log

In [ ]:
consolidated_transactions = consolidate(cleaned_transactions)
consolidated_transactions

# Count old SKUs and new canonicals

In [ ]:
#poptart transaction count for each sku
groupbySku = consolidated_transactions.groupby('sku').agg(
    count = ('sku', 'count')
).reset_index()
poptarts = ["SKU#459812", "POPTART-STRAW", "PT-12", "PT_STRAWBERRY"]

groupbySkuFiltered = groupbySku[groupbySku['sku'].isin(poptarts)]

#poptart transaction count for each canonical
groupbyCanonical = consolidated_transactions.groupby('canonical_sku').agg(
    count = ('canonical_sku', 'count')
).reset_index()

groupbyCanonicalFiltered = groupbyCanonical[groupbyCanonical['canonical_sku'].isin(poptarts)]

print(groupbySkuFiltered)
print(groupbyCanonicalFiltered)

# Bar chart

In [ ]:
from matplotlib import pyplot as plt

fig, ax = plt.subplots(ncols=2)
sns.barplot(data = groupbySkuFiltered,x = 'sku', y = 'count', ax = ax[0])
sns.barplot(data = groupbyCanonicalFiltered,x = 'canonical_sku', y = 'count', ax = ax[1])
ax[0].set_ylim(0, 2500)
ax[1].set_ylim(0, 2500)
plt.show()

The true daily sales volume after standardization is 2053. Upon first glance for fragmented sales is that it looks like there was around 500 sales for Pop-Tarts, and therefore not a good organization to determine the actual total sales for Pop-Tarts. With the cleaned/consolidated view, visually there is more sales above 2000 and cleaner to look at since all the sales are totaled. As for the business decision for consolidated, it would be encouraged to send more Pop-Tarts due to it's high sales compared to fragmented where the business would either keep it consistent or rescind Pop-Tart sales.

### Question 3: Weather Correlation Analysis

> Using weather data pulled from your chosen free API, how do specific weather indicators (wind speed, precipitation, barometric pressure) correlate with daily sales volume across categories? Is there a detectable lag between weather severity escalation and purchasing spikes?

**Requires:** Weather API integration, timestamp/timezone alignment, correlation analysis (scatter plot, heatmap, or time-series overlay), lag analysis discussion.

---

# API call function

In [ ]:
import time
import requests

BASE = 'https://archive-api.open-meteo.com/v1/archive'
RETRY_STATUSES = {429, 500, 502, 503, 504}
FALLBACK = pd.DataFrame({
    'date': pd.date_range('2024-09-01', periods=30),
    'precipitation_sum': [3.0, 12.4, 24.9, 22.1, 23.1, 9.3, 6.8, 94.7, 28.0, 14.4,
                          12.0, 0.7, 0.6, 21.1, 1.2, 7.9, 2.3, 13.3, 1.4, 0.7,
                          0.4, 2.2, 1.3, 0.3, 6.4, 15.2, 2.5, 18.0, 4.6, 3.3],
    'windspeed_10m_max': [18.8, 13.5, 22.8, 15.5, 15.2, 9.0, 14.2, 12.6, 11.5, 14.3,
                          15.8, 12.1, 10.4, 17.4, 9.0, 13.3, 12.4, 9.3, 12.9, 15.3,
                          19.9, 17.9, 16.6, 17.7, 27.2, 48.8, 45.1, 19.8, 11.7, 8.6],
    'source': 'orlando fallback',
})

def daily_weather(start, end, lat, lon, tries=3):
    params = {
        'latitude': lat, 'longitude': lon,
        'start_date': start, 'end_date': end,
        'daily': 'precipitation_sum,windspeed_10m_max,pressure_msl_mean',
        'timezone': 'America/New_York',
    }
    for attempt in range(1, tries + 1):
        try:
            r = requests.get(BASE, params=params, timeout=10)
        except (requests.Timeout, requests.ConnectionError):
            if attempt == tries:
                raise
            time.sleep(2 ** attempt)
            continue
        if r.status_code in RETRY_STATUSES:
            if attempt == tries:
                r.raise_for_status()
            time.sleep(2 ** attempt)
            continue
        r.raise_for_status()
        frame = pd.DataFrame(r.json()['daily'])
        frame = frame.rename(columns={'time': 'date'})
        frame['date'] = pd.to_datetime(frame['date'])
        frame['source'] = 'open-meteo'
        return frame

# Attempt call...

In [ ]:
try:
    weather_orlando = daily_weather('2024-09-01', '2024-09-30', lat=28.54, lon=-81.38)
except requests.RequestException as err:
    print('API call failed:', err)
    weather_orlando = FALLBACK

weather_orlando

# Grab category count data and join

In [ ]:
category_count = cleaned_transactions.groupby(["date", "category"]).agg(
    count=('category', 'count')
).reset_index()

# merge

merged_df = pd.merge(category_count, weather_orlando, on='date', how='left')
merged_df

# Chart showing correlation between weather data and breakfast sales surge

In [ ]:
# Breakfast only
df_breakfast = merged_df[merged_df['category'] == 'breakfast'][[
    'date',
    'count',
    'precipitation_sum',
    'windspeed_10m_max',
    'pressure_msl_mean'
]].copy()

df_breakfast = df_breakfast.rename(columns={'count': 'breakfast_sales'})

dates_str = df_breakfast['date'].dt.strftime('%Y-%m-%d')

# chart

fig, axes = plt.subplots(4, 1, figsize=(12, 12), sharex=True)

metrics = [
    ('breakfast_sales', 'Transactions in Breakfast Category', 'blue'),
    ('precipitation_sum', 'Precipitation Sum (mm)', 'orange'),
    ('windspeed_10m_max', 'Max Wind Speed 10m (km/h)', 'green'),
    ('pressure_msl_mean', 'Mean Sea Level Pressure', 'red')
]

for i, (col, title, color) in enumerate(metrics):
    axes[i].bar(dates_str, df_breakfast[col], color=color, alpha=0.7)
    axes[i].set_title(title, fontsize=11, fontweight='bold')
    axes[i].grid(axis='y', linestyle='--', alpha=0.5)

plt.xlabel('Date', fontsize=12)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


Barometic pressure and precipitation don't seem to have a correlation with breakfast sales. However wind speed does. You can see the sales spike before the wind speed does.

### Question 4: Store-Level Geographic Patterns

> Do all Florida store locations experience the same surge patterns, or do stores closer to the predicted hurricane path show earlier and larger spikes? Identify any stores that appear to be outliers.

**Requires:** Joining store location data with transactions, geographic grouping, comparing surge timing and magnitude across stores, bar chart or grouped comparison, outlier identification.

---

### Question 5: Signal vs. Artifact — The Board Games Investigation

> The board games/entertainment category shows a +250% sales increase. Using the SQLite inventory and sales database, investigate whether this is a genuine demand signal or a data artifact (e.g., SKU miscoding, category lumping, inventory reporting lag). Present your evidence and make a recommendation: should Walmart stock extra board games for the next storm?

**Requires:** Querying the SQLite database with `sqlite3` + `pandas`, cross-referencing with the transaction CSV, SKU/category audit, evidence-based written argument, supporting visualization.

---

# Pull SQL data and convert to dataframe

In [ ]:
import sqlite3

connection = sqlite3.connect("../data/inventory_and_sales.db")

inventory_levels = pd.read_sql_query("SELECT * FROM inventory_levels", connection)
daily_sales_summary = pd.read_sql_query("SELECT * FROM daily_sales_summary", connection)
print(inventory_levels.head())
print(daily_sales_summary.head())

In [ ]:
# Pull only entertainment transaction data

transactions_entertainment = consolidated_transactions[consolidated_transactions['category'] == 'entertainment']

transactions_entertainment

In [ ]:
print(f"unique skus: {transactions_entertainment['sku'].unique()}")
entertainment_skus = transactions_entertainment['sku'].unique().to_numpy()
print(f"unique canonical skus: {transactions_entertainment['canonical_sku'].unique()}")

# Examine the stock of each SKU

In [ ]:
entertainment_stock = inventory_levels[inventory_levels['sku'].isin(entertainment_skus)]

# oops need to turn it into an actual date
entertainment_stock['date'] = pd.to_datetime(entertainment_stock['date'])
entertainment_stock = entertainment_stock.sort_values('date')

fig, ax = plt.subplots(figsize=(12, 6))

sns.lineplot(data=entertainment_stock, x='date', y='units_on_hand', hue='sku', errorbar=None, ax=ax)
plt.xticks(rotation=45)
plt.show()

# Examine the transactions per day on these SKUs from the CSV

In [ ]:
# group by day
entertainment_transactions_per_day = transactions_entertainment.groupby(['date', 'sku']).agg(
    sales=('quantity', 'sum')
)


fig, ax = plt.subplots(figsize=(12, 6))

sns.lineplot(data=entertainment_transactions_per_day, x='date', y='sales', hue='sku', errorbar=None, ax=ax)
plt.xticks(rotation=45)
plt.show()

yeah that's kind of weird, there's a drop in stock on the 21st but the transaction surge is on the 25th

# What about the daily sales summary from the database?

In [ ]:
daily_sales_summary['date'] = pd.to_datetime(daily_sales_summary['date'], format='mixed')
entertainment_sales = daily_sales_summary[daily_sales_summary['category'] == "Entertainment"]
entertainment_sales = entertainment_sales.groupby('date').agg(
    sales=('total_units', 'sum')
)

entertainment_stock_per_day = entertainment_stock.groupby('date').agg(
    stock=('units_on_hand', 'sum')
)

merged_df = pd.merge(entertainment_sales, entertainment_stock_per_day, on='date', how='outer').reset_index()

long_df = pd.melt(merged_df, id_vars='date', value_vars=["sales", "stock"], value_name='units', var_name='type')

fig, ax = plt.subplots(figsize=(12, 6))

sns.lineplot(data=long_df, x='date', y='units', hue='type', errorbar=None, ax=ax)
plt.xticks(rotation=45)
plt.show()

**Q5 answer:** It seems like there was a genuine increase in demand for monopoly and card games during the hurricane. It was not an artifact, because you can clearly see a sharp decrease in stock and an increase in sales on the 21st. Does this mean that Walmart should continue to stock more entertainment products? Maybe, but I'm leaning towards not too many more, as long as it doesn't displace emergency supply space.